# Handwriting — the full, self-study version

### AI Builders Lab · take-home companion to Project 2

This is the long version of the notebook we ran in class. Everything from Project 2 is
here, plus three things we did not have time for: the digits the model gets **wrong**,
uploading a **photo** of real pen-on-paper handwriting, and an experiment section.

Work through it at your own pace.

By the end of this notebook you will have:

1. Built an **Artificial Neural Network (ANN)** from a blank page
2. Trained it on **60,000 handwritten digits** written by real people
3. Made it read **a digit that you write yourself**, right here in the browser

You do not need to install anything. You do not need a powerful computer.
Everything runs on Google's machines, for free, in about 5 minutes of computing time.

---

### How to use this notebook

- Each grey box below is a **code cell**. Click it and press **Shift + Enter** to run it.
- A cell that is running shows a spinning circle. A cell that has finished shows a number, like `[1]`.
- **Run the cells in order, from top to bottom.** Later cells depend on earlier ones.
- If something breaks, don't panic — scroll to **Part 9: When Things Go Wrong** at the end.

### First, save your own copy

Go to **File → Save a copy in Drive**. Otherwise your changes disappear when you close the tab.
Work in *your* copy, not this one.

---
# Part 1 — Open the toolbox

Python by itself does not know what a neural network is. We have to *import* the tools.

Think of it like a lab bench. Before an experiment you lay out the equipment you need:

| Tool | What it does for us |
|---|---|
| `tensorflow` / `keras` | Builds and trains the neural network. This is the engine. |
| `numpy` | Handles arrays of numbers. Images are just arrays of numbers. |
| `matplotlib` | Draws pictures and graphs so we can *see* what is happening. |
| `PIL` | Opens and resizes image files. |

Colab already has all of these installed, so we only import them — no `pip install` needed.

In [ ]:
# Run this cell first (Shift + Enter)

import numpy as np                    # arrays of numbers
import matplotlib.pyplot as plt       # drawing graphs and images
import tensorflow as tf               # the neural network engine
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

# Fix the random seed so your results match your classmates' roughly.
# Neural networks start with RANDOM weights, so without this, everyone
# gets slightly different numbers every single run.
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("Toolbox is open. You are ready to go.")

---
# Part 2 — Get the data

A neural network is not programmed. It is **trained**. It learns by looking at thousands of examples,
the way a child learns what a "7" looks like by seeing many sevens.

So we need examples. We will use **MNIST** — a famous dataset of 70,000 handwritten digits,
collected from US Census Bureau employees and high school students in the 1990s.
Every image is 28 x 28 pixels, in greyscale, and comes with a label saying which digit it really is.

MNIST is built into Keras, so downloading it takes one line.

### Why do we split the data into "train" and "test"?

This is one of the most important ideas in all of machine learning.

If you give a student the answer key before the exam, a perfect score proves nothing —
you cannot tell whether they *learned* or just *memorised*. Same with a neural network.

So we hide 10,000 images from the network during training. It never sees them.
Then we test on those. **The test score is the only honest measure of whether the network really learned.**

In [ ]:
# Download the data. First time takes ~5 seconds.
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

# X = the images (the questions).  Y = the correct digit (the answers).
print("Training images:", X_train.shape)   # (60000, 28, 28) -> 60000 images, each 28x28 pixels
print("Training labels:", Y_train.shape)   # (60000,)        -> 60000 answers
print("Testing images: ", X_test.shape)    # (10000, 28, 28) -> the hidden exam
print("Testing labels: ", Y_test.shape)

print("\nThe first 10 answers are:", Y_train[:10])

---
# Part 3 — Look at the data before you model it

**Never train a model on data you have not looked at.** This is a habit worth building now.
Half of all machine learning bugs are actually data bugs, and you find them by looking.

Let's see the first 10 images with their labels.

In [ ]:
# Show the first 10 training images side by side
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_train[i], cmap='gray')   # cmap='gray' because these are greyscale, not colour
    plt.title(Y_train[i])                 # put the correct answer on top
    plt.axis('off')                       # hide the pixel-number axes, they add nothing here
plt.suptitle("What the network is about to study")
plt.show()

### Now the important part: a picture *is* a table of numbers

To a computer there is no such thing as an image. There is only a grid of numbers.
Each number is one pixel's brightness, from **0 (pure black)** to **255 (pure white)**.

Run the cell below and then squint at the output. You will see the shape of the digit
appear in the numbers — the zeros are the empty paper, the big numbers are the ink.

**This is the whole trick of computer vision.** Once an image is numbers, it is just arithmetic.

In [ ]:
# Print image number 0 as raw numbers.
# We set the print width wide so the 28 columns line up in a square.
np.set_printoptions(linewidth=200)

print("This image is labelled:", Y_train[0])
print("\nThe same image as raw pixel values (0 = black paper, 255 = bright ink):\n")
print(X_train[0])

In [ ]:
# Same thing, easier to read: print a dot for empty paper and a # for ink.
# Now you can literally read the digit out of the text.
for row in X_train[0]:
    print("".join("#" if pixel > 128 else ("+" if pixel > 40 else ".") for pixel in row))

---
# Part 4 — Prepare the data (normalization)

Right now our pixels run from 0 to 255. We are going to squeeze them into the range **0 to 1**
by dividing every pixel by 255. This is called **normalization**.

### Why bother?

A neural network learns by nudging its internal weights up and down in small steps.
If the inputs are huge (255) the nudges become huge and unstable — the network overshoots,
the error bounces around, and training either takes forever or fails completely.
Keeping inputs small and consistent keeps the learning steps well behaved.

Rule of thumb you can carry into every future project:
**neural networks like small, similarly-sized inputs.**

Notice we do the *exact same* transformation to the test set. Whatever you do to the training
data, you must do to anything you later predict on — including your own handwriting at the end.

In [ ]:
# Turn 0-255 into 0.0-1.0
X_train = X_train / 255.0
X_test  = X_test  / 255.0

print("Smallest pixel value now:", X_train.min())
print("Largest pixel value now: ", X_train.max())

---
# Part 5 — Build the neural network

Now we design the brain. We are using a `Sequential` model, which means
**layers stacked one after another**, information flowing straight through from top to bottom.

Here is our architecture and the reason for each piece:

| Layer | Size | Why |
|---|---|---|
| `Flatten` | 28x28 → 784 | A Dense layer wants a single row of numbers, not a square. So we unroll the 28 rows of pixels into one long line of 784 values. |
| `Dense(128, relu)` | 128 neurons | The main workhorse. Every one of the 784 pixels connects to every one of these 128 neurons — that is what "Dense" means. These neurons learn to detect strokes, curves, and loops. |
| `Dropout(0.2)` | — | During training, randomly switch off 20% of the neurons each round. Sounds destructive, but it stops the network from leaning too hard on any one neuron. It prevents **overfitting** (memorising instead of learning). |
| `Dense(64, relu)` | 64 neurons | A second, narrower layer that combines the simple strokes into bigger patterns. |
| `Dense(10, softmax)` | 10 neurons | The answer layer. One neuron per digit, 0 through 9. |

### Why `relu` on the middle layers?

ReLU means "if the number is negative, make it zero; otherwise leave it alone."
It is almost absurdly simple, and that is exactly why it works — it is fast to compute,
and it lets the network build up non-linear shapes. Without something like ReLU,
stacking layers would be pointless: a stack of pure straight-line layers collapses
into one straight line, and a straight line cannot recognise a handwritten 8.

### Why `softmax` on the last layer?

Softmax turns the 10 raw output numbers into **10 probabilities that add up to 1.0**.
So instead of a meaningless score, we get an answer like:
*"I'm 97% sure this is a 3, 2% sure it's an 8, 1% everything else."*
That confidence number is genuinely useful — you will see later that when the model is wrong,
it is usually also unsure, and you can spot bad predictions by watching for low confidence.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28)),              # tell the model what one image looks like
    layers.Flatten(),                          # 28x28 square -> 784 numbers in a row
    layers.Dense(128, activation='relu'),      # hidden layer 1: 128 neurons
    layers.Dropout(0.2),                       # randomly ignore 20% of neurons while training
    layers.Dense(64,  activation='relu'),      # hidden layer 2: 64 neurons
    layers.Dense(10,  activation='softmax'),   # output: probability for each digit 0-9
], name="handwriting_ANN")

model.summary()

### Read the summary above — the parameter count is the point

Look at `Total params`. That is roughly **109,000 numbers** the network will adjust while learning.
Nobody sets those by hand. Training is the process of the computer finding good values for all
109,000 of them, automatically, by trial and error.

Check the first Dense layer: 100,480 parameters. Where does that come from?

$$784 \text{ pixels} \times 128 \text{ neurons} = 100{,}352 \text{ weights}, \quad + \; 128 \text{ biases} = 100{,}480$$

Every connection gets its own weight. That is the whole model — a very large pile of multiplications.

---
# Part 6 — Compile: set the rules of learning

Before training we tell the model three things:

**1. `optimizer='adam'` — HOW to learn.**
Adam ("adaptive moment estimation") decides how big a step to take when adjusting each weight.
Big steps early when it is badly wrong, small careful steps later when it is close. It is the
sensible default for almost everything, and we use it throughout this course.

**2. `loss='sparse_categorical_crossentropy'` — HOW WRONG it is.**
The loss function is the score the network is trying to *minimise*. It measures the gap between
the predicted probabilities and the truth. "Sparse categorical" is the version to use when your
labels are plain integers (`7`) rather than lists (`[0,0,0,0,0,0,0,1,0,0]`). Ours are plain
integers, so: sparse.

**3. `metrics=['accuracy']` — how WE want to judge it.**
Loss is what the machine optimises; accuracy (percent correct) is what humans understand.
This one is just for our benefit — it does not affect training at all.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled. The rules of learning are set.")

---
# Part 7 — Train it

This is the moment the learning actually happens.

- **`epochs=10`** — go through all 60,000 images 10 times. One full pass = one epoch.
  The network improves a little on each pass, like re-reading a textbook chapter.
- **`batch_size=128`** — look at 128 images, then update the weights. Then the next 128.
  Updating after every single image would be painfully slow; updating only after all 60,000
  would be too coarse. 128 is a good middle ground.
- **`validation_data`** — after each epoch, quietly check the score on the hidden test set
  so we can watch for overfitting in real time.

Run the cell. It takes about **30–60 seconds**. Watch the numbers climb.

**What to watch for:** `accuracy` is the score on material it is studying.
`val_accuracy` is the score on the hidden exam. You want *both* going up.
If training accuracy keeps climbing while validation accuracy stalls or drops,
the network has started memorising rather than learning — that is overfitting.

In [ ]:
history = model.fit(
    X_train, Y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_test, Y_test),   # the hidden exam, checked after each epoch
    verbose=1                           # 1 = show me the progress bar
)

print("\nTraining finished.")

---
# Part 8 — How well did it do?

First the headline number: accuracy on the 10,000 images it has **never seen**.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=0)

print("Loss on unseen data:     ", round(test_loss, 4))
print("ACCURACY on unseen data: ", round(test_accuracy * 100, 2), "%")
print("\nOut of 10,000 digits it had never seen, it got about",
      int(test_accuracy * 10000), "right.")

You should be somewhere around **97–98%**.

Sit with that for a second. We wrote about fifteen lines of model code, and a program
that was never told a single rule about what digits look like — no "a 7 has a horizontal bar",
no "an 8 has two loops" — now reads human handwriting better than most people would guess.
It figured out all of it from examples alone.

Now let's plot the learning, because a graph tells you things a single number cannot.

In [ ]:
plt.figure(figsize=(12, 4))

# Left: accuracy going up
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='training')
plt.plot(history.history['val_accuracy'], label='unseen test data')
plt.title('Accuracy (higher is better)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)

# Right: loss coming down
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='training')
plt.plot(history.history['val_loss'], label='unseen test data')
plt.title('Loss (lower is better)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Reading these graphs

Look at the gap between the two lines on each graph.

- **Lines close together** → the network is genuinely learning. Good.
- **Training line much better than test line** → overfitting. It is memorising the 60,000
  training images instead of learning what digits look like.
- **Test loss turning upward while training loss keeps falling** → you have trained too long.
  That upward turn is the moment memorising took over. This is exactly why we kept `epochs`
  modest and added `Dropout`.

Look at your own graph. Where does the improvement flatten out? That tells you whether
running more epochs would have been worth the time.

### Look at a single prediction in detail

Remember softmax gives us 10 probabilities, not one answer. Let's see them.

In [ ]:
# Ask the model about every test image at once
predictions = model.predict(X_test, verbose=0)

# Pick one image to inspect. CHANGE THIS NUMBER and re-run to look at others.
i = 9

probabilities = predictions[i]
guess = np.argmax(probabilities)      # argmax = "which position holds the biggest number?"

plt.figure(figsize=(10, 3.5))

plt.subplot(1, 2, 1)
plt.imshow(X_test[i], cmap='gray')
plt.title("True answer: " + str(Y_test[i]))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.bar(range(10), probabilities * 100)
plt.xticks(range(10))
plt.xlabel('Digit')
plt.ylabel('Confidence (%)')
plt.title("Model says: " + str(guess) + "  (" + str(round(probabilities[guess]*100, 1)) + "% sure)")
plt.ylim(0, 100)

plt.tight_layout()
plt.show()

### Now the more interesting question: what does it get WRONG?

Successes teach you little. Failures teach you a lot.
Let's pull out the digits the model missed and look at them.

In [ ]:
guesses = np.argmax(predictions, axis=1)     # the model's answer for every test image
wrong = np.where(guesses != Y_test)[0]       # positions where it was wrong

print("The model got", len(wrong), "out of 10,000 wrong.\n")

plt.figure(figsize=(14, 3.5))
for n, i in enumerate(wrong[:10]):
    plt.subplot(1, 10, n + 1)
    plt.imshow(X_test[i], cmap='gray')
    plt.title("is " + str(Y_test[i]) + "\nsaid " + str(guesses[i]), fontsize=9)
    plt.axis('off')
plt.suptitle("Ten mistakes")
plt.show()

**Look at these carefully — this is a real discussion, not a formality.**

Some of these you would probably get wrong too. Some are genuinely ambiguous scrawls.
A few are arguably mislabelled in the original dataset.

This matters because it sets a realistic expectation for every AI system you will ever build:
**the ceiling on accuracy is usually the data, not the model.** If humans disagree about
what the answer is, no neural network is going to hit 100%.

---
### Save your trained model

Training took a minute here, but real models can take days. You never want to retrain
something you already have. `model.save()` writes all 109,000 learned numbers to a file.

The file lands in Colab's temporary storage — click the **folder icon** in the left sidebar
to see it. Note that Colab wipes this storage when your session ends, so download it
(right-click → Download) if you want to keep it.

In [ ]:
model.save('handwriting_model.keras')
print("Saved to handwriting_model.keras")

# To load it back later — in a different notebook, next week, anywhere:
# from tensorflow import keras
# model = keras.models.load_model('handwriting_model.keras')

---
---
# Part 9 — THE FUN PART: make it read *your* handwriting

Everything up to here used somebody else's data. Now we point the model at you.

But there is a problem to solve first, and it is the single most important practical lesson
in this notebook.

### The model is extremely fussy about its input

Our model has only ever seen images that look like this:

- exactly **28 x 28** pixels
- **white ink on a black background** (not black ink on white paper!)
- the digit **centred**, and sized to fill about 20 of the 28 pixels
- pixel values between **0 and 1**

Your drawing or your photo will be none of those things. It will be big, colour, black-on-white,
off-centre, and possibly crooked.

If you feed it in raw, the model will confidently give you garbage.
**It will not throw an error. It will just be wrong.** That is what makes this failure mode dangerous.

So we write a function that converts anything into MNIST's format. Read the comments —
each step exists because of a specific way the prediction fails without it.

In [ ]:
from PIL import Image

def prepare_digit(img, show=True):
    """Convert ANY picture of a single digit into the 28x28 format the model expects."""

    original = img
    g = np.array(img.convert("L"), dtype=np.float32)   # "L" = convert to greyscale

    # STEP 1: Flip black-on-white to white-on-black.
    # MNIST is white ink on black paper. Your notebook paper is the opposite.
    # We check the brightness of the border pixels: if the edge of the picture is
    # light, we are looking at paper, so we invert.
    border = np.concatenate([g[0, :], g[-1, :], g[:, 0], g[:, -1]])
    if border.mean() > 127:
        g = 255.0 - g

    # STEP 2: Stretch the contrast and delete the faint stuff.
    # Photos have shadows, paper texture, and grey smudges. Anything dimmer than
    # 40% of the brightest ink is treated as background and set to pure black.
    g = g - g.min()
    if g.max() > 0:
        g = g / g.max() * 255.0
    g[g < 0.4 * g.max()] = 0

    # STEP 3: Crop away the empty space, keeping only the ink.
    ys, xs = np.nonzero(g)
    if len(ys) == 0:
        print("I can't find any ink in this image. Try drawing darker or thicker.")
        return np.zeros((28, 28), dtype=np.float32)
    g = g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    # STEP 4: Resize so the longest side is 20 pixels.
    # Why 20 and not 28? Because the people who built MNIST scaled every digit into
    # a 20x20 box and left a 4-pixel margin. We copy them exactly. Get this wrong
    # and your digit is the wrong size compared to everything the model studied.
    h, w = g.shape
    scale = 20.0 / max(h, w)
    new_h, new_w = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    g = np.array(Image.fromarray(g.astype(np.uint8)).resize((new_w, new_h), Image.LANCZOS),
                 dtype=np.float32)

    # STEP 5: Paste into a 28x28 black square, centred by CENTRE OF MASS.
    # MNIST centres each digit on its centre of gravity, not its bounding box.
    # This is the step everyone skips, and skipping it is the #1 reason
    # "it works on MNIST but not on my own writing".
    canvas = np.zeros((28, 28), dtype=np.float32)
    top, left = (28 - new_h) // 2, (28 - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = g
    cy, cx = np.array(np.nonzero(canvas)).mean(axis=1)
    canvas = np.roll(canvas, int(round(13.5 - cy)), axis=0)
    canvas = np.roll(canvas, int(round(13.5 - cx)), axis=1)

    # STEP 6: Scale to 0-1, exactly as we did to the training data.
    canvas = canvas / 255.0

    if show:
        plt.figure(figsize=(7, 3))
        plt.subplot(1, 2, 1); plt.imshow(original, cmap='gray')
        plt.title("What you gave it"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(canvas, cmap='gray')
        plt.title("What the model actually sees"); plt.axis('off')
        plt.show()

    return canvas


def predict_digit(img28):
    """Feed a prepared 28x28 image to the model and show the verdict."""
    p = model.predict(img28.reshape(1, 28, 28), verbose=0)[0]
    guess = int(np.argmax(p))

    print("=" * 44)
    print("   THE MODEL SAYS:", guess, "  (", round(p[guess] * 100, 1), "% confident )")
    print("=" * 44)
    print("\nIts full opinion:")
    for digit in range(10):
        bar = "#" * int(p[digit] * 40)
        print(f"  {digit} | {bar:<40} {p[digit]*100:5.1f}%")
    return guess

print("Helper functions ready.")

## 9a — Draw a digit with your mouse

Run the cell below. A black box appears. **Draw a single digit in it with your mouse or trackpad**
(hold the button down and drag), then click **DONE**.

Tips for a good result:
- Draw **big** — fill most of the box.
- Draw **thick and confidently**. Thin, scratchy lines vanish when we shrink to 28x28.
- One digit only.
- Made a mess? Click **Clear** and start over.

In [ ]:
from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
import io

canvas_html = '''
<div style="font-family: sans-serif;">
  <canvas id="pad" width="280" height="280"
          style="border:3px solid #555; background:#000; cursor:crosshair; touch-action:none;"></canvas>
  <br><br>
  <button id="done"  style="font-size:16px; padding:8px 18px; cursor:pointer;">DONE - read my digit</button>
  <button id="clear" style="font-size:16px; padding:8px 18px; cursor:pointer;">Clear</button>
</div>
<script>
  var c   = document.getElementById('pad');
  var ctx = c.getContext('2d');
  ctx.fillStyle = 'black';
  ctx.fillRect(0, 0, c.width, c.height);
  ctx.strokeStyle = 'white';
  ctx.lineWidth   = 20;          // thick, so the stroke survives shrinking to 28x28
  ctx.lineCap     = 'round';
  ctx.lineJoin    = 'round';

  var drawing = false;
  function spot(e) {
    var r = c.getBoundingClientRect();
    return [e.clientX - r.left, e.clientY - r.top];
  }
  c.addEventListener('pointerdown', function(e) {
    drawing = true;
    var p = spot(e);
    ctx.beginPath();
    ctx.moveTo(p[0], p[1]);
    ctx.lineTo(p[0], p[1]);
    ctx.stroke();
  });
  c.addEventListener('pointermove', function(e) {
    if (!drawing) return;
    var p = spot(e);
    ctx.lineTo(p[0], p[1]);
    ctx.stroke();
  });
  window.addEventListener('pointerup', function() { drawing = false; });

  document.getElementById('clear').onclick = function() {
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, c.width, c.height);
  };

  // Python waits on this promise until you click DONE
  var data = new Promise(function(resolve) {
    document.getElementById('done').onclick = function() {
      resolve(c.toDataURL('image/png'));
    };
  });
</script>
'''

display(HTML(canvas_html))
drawing_data = eval_js("data")                       # pauses here until you click DONE
raw = b64decode(drawing_data.split(',')[1])
my_drawing = Image.open(io.BytesIO(raw))

print("Got your drawing.\n")

In [ ]:
# Now clean it up and ask the model
prepared = prepare_digit(my_drawing)
predict_digit(prepared)

**Did it get it right?**

Go back, draw a different digit, and run both cells again. Try to find one it fails on.

When it does fail, look at the right-hand picture — *what the model actually sees*.
Nine times out of ten the failure is visible right there: the stroke got too thin,
or the digit was tiny in a corner, or you drew a shape genuinely unlike MNIST's style
(a European 7 with a crossbar, a 1 with a big flag and a base serif, an open-topped 4).

That is not a bug. That is the model honestly telling you it has never seen handwriting like yours.
**A model can only recognise what it has been shown.** Remember that sentence — it explains most
AI failures you will read about in the news.

## 9b — Or use a photo of real handwriting

Prefer pen and paper? Do this:

1. Write **one large digit** on white paper with a **thick dark pen or marker** (not pencil — too faint).
2. Photograph it straight on, in good light, and **crop tightly** around the digit.
3. Run the cell below and click **Choose Files** to upload it.

The same `prepare_digit` function handles it — that is why we wrote it to be general.

In [ ]:
from google.colab import files

uploaded = files.upload()          # opens a file picker

for filename in uploaded.keys():
    print("\n--- " + filename + " ---")
    photo = Image.open(io.BytesIO(uploaded[filename]))
    prepared = prepare_digit(photo)
    predict_digit(prepared)

---
# Part 10 — Experiment (do this in class)

You now have a working model. The best way to understand it is to break it.

Copy the model-building and training cells into the empty cell below, change **one thing**,
re-run, and write down what happened to the test accuracy. One change at a time —
if you change three things at once you learn nothing about any of them.

Things worth trying:

| Change | Question it answers |
|---|---|
| `Dense(128)` → `Dense(16)` | How small can the brain get before it starts failing? |
| `Dense(128)` → `Dense(512)` | Does a bigger brain always help? (It costs training time — is it worth it?) |
| Delete the `Dropout` layer | Watch the gap between training and test accuracy. Does overfitting appear? |
| `epochs=10` → `epochs=1` | How much does it know after a single pass? |
| `epochs=10` → `epochs=40` | Does it keep improving forever? Where does the test loss turn upward? |
| `learning_rate=0.001` → `0.5` | What does a wildly-too-large learning rate look like? (Spoiler: it is dramatic.) |
| Add a third `Dense(64, relu)` | Do more layers beat wider layers? |
| Remove all `activation='relu'` | Why did we insist non-linearity was essential? See for yourself. |

In [ ]:
# Your experiment space. Paste, change ONE thing, re-run, record the result.

---
---
# HOMEWORK

Submit one notebook (`File → Download → .ipynb`) plus a short written answer for each part.
Your code must run top to bottom without errors.

### Problem 1 — The architecture study *(30 points)*

Train **five** different versions of the network, changing one thing each time, and fill in this table:

| # | What I changed | Params | Training accuracy | Test accuracy | Time to train |
|---|---|---|---|---|---|
| 1 | nothing (the baseline above) | 109,386 | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |
| 5 | | | | | |

Then answer in a few sentences: **which change helped most, and which change cost the most
training time for the least benefit?** In engineering you are always trading accuracy against
cost — say what trade you would make and why.

### Problem 2 — Test it on *your* handwriting *(30 points)*

Write out all ten digits, 0 through 9, in your own hand. Feed each one through the model
(canvas or photo, your choice).

- Report your model's accuracy on **your** 10 digits.
- Compare it to the ~97.7% it scored on MNIST.
- **The gap is the assignment.** Explain it. Why does a model that reads 9,770 out of 10,000
  strangers' digits correctly stumble on yours? Name at least two specific causes and support
  each one with a screenshot of the "what the model actually sees" picture.

### Problem 3 — Failure analysis *(25 points)*

Find **three** digits the model gets wrong — from the test set, your own handwriting, or both.

For each one:

1. Show the image and the model's full confidence breakdown.
2. Was it *confidently* wrong (>90%) or *uncertain* wrong (<60%)?
3. Explain what you think caused it.

Then answer: **which is more dangerous in a real system — a model that is wrong and knows it,
or a model that is wrong and certain?** Give a real-world example where the difference matters.

### Problem 4 — Stretch, optional *(15 bonus points)*

Our network flattens the image into 784 unrelated numbers on the very first line —
which throws away every fact about which pixels sit next to which. A **Convolutional**
network keeps that spatial information. Try replacing the first layers with:

```python
layers.Input(shape=(28, 28, 1)),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dense(64, activation='relu'),
layers.Dense(10, activation='softmax'),
```

Report the new test accuracy and how long it took to train. Was the extra time worth it?
(We will cover CNNs properly later in the course — this is a preview.)

### What to hand in

- Your `.ipynb` file with all cells run and outputs visible
- Answers written in **markdown cells inside the notebook** (Insert → Text cell)
- Screenshots where the question asks for them

---
# Part 11 — When things go wrong

Errors are normal. Every practicing engineer reads error messages all day. Here are the ones
this notebook produces most often.

**`NameError: name 'model' is not defined`**
You skipped a cell, or the session restarted. Run **Runtime → Run all** and wait.

**`NameError: name 'X_train' is not defined`**
Same cause. The data cell has to run before anything that uses the data.

**The accuracy is stuck around 10%**
10% is pure guessing — 1 chance in 10. Almost always this means you ran the normalization cell
(`X_train = X_train / 255.0`) **twice**, so your pixels are now in the range 0 to 0.004. Fix it by
**Runtime → Restart session**, then run everything once, in order.

**My drawing gets predicted wrong every single time**
Look at the "what the model actually sees" picture on the right.

- Is it blank or nearly blank? Draw thicker and darker.
- Is the digit tiny or off in a corner? Something went wrong in cropping — redraw larger.
- Does it look like a reasonable digit but still gets the wrong answer? Then it is a genuine
  disagreement, and it belongs in Homework Problem 3. That is a result, not a bug.

**The photo upload gives nonsense**
Usually lighting. Crop tightly around the digit, use a marker rather than a pencil, avoid shadows
falling across the paper, photograph straight on rather than at an angle.

**`Your session crashed after using all available RAM`**
You probably set a huge layer size or batch size. Restart and go back to sensible numbers.

**Everything is broken and I don't know why**
**Runtime → Restart session and run all.** This fixes a genuinely surprising share of problems,
because it clears out any half-finished state from cells you ran out of order.

---

### What you did today

You built a neural network from nothing, trained it on 60,000 examples, got it to about 98%
on data it had never seen, and made it read your own handwriting.

More importantly, you met the ideas that every project in this course rests on:
**train/test splits, normalization, layers and weights, activation functions, overfitting and
dropout, loss versus accuracy, and the hard truth that a model can only recognise what it has
been shown.**

Everything after this — CNNs, LSTMs, voice, language, reinforcement learning — is a variation
on what you just did.